Please do not run this notebook, this barely ran even with 64gb of ram

## Load data

In [1]:
import pandas as pd
clean_news_data = pd.read_pickle('data/labelled_dtm.pkl')

clean_news_data_ctx = pd.read_pickle('data/labelled_tfidf_dtm.pkl')

In [2]:
clean_news_data.head()

,aaa,aafter,aaliyah,aaplo,aaplus,aaron,aaroncovfefe,aaronshhh,aarp,ab,...,zuker,zukunft,zuppello,zyklon,zypries,zz,zztaine,zzzzaaaacccchhh,zzzzzzzzzzzzz,real_label
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [3]:
clean_news_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5182 entries, 0 to 5181
Columns: 53541 entries, aaa to real_label
dtypes: int64(53541)
memory usage: 2.1 GB


In [4]:
clean_news_data['real_label'].value_counts()

real_label
1    2591
0    2591
Name: count, dtype: int64

In [5]:
clean_news_data_ctx.head()

,aaa,aafter,aaliyah,aaplo,aaplus,aaron,aaroncovfefe,aaronshhh,aarp,ab,...,zuker,zukunft,zuppello,zyklon,zypries,zz,zztaine,zzzzaaaacccchhh,zzzzzzzzzzzzz,real_label
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


In [6]:
clean_news_data_ctx.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5182 entries, 0 to 5181
Columns: 53541 entries, aaa to real_label
dtypes: float64(53540), int64(1)
memory usage: 2.1 GB


In [7]:
clean_news_data_ctx['real_label'].value_counts()

real_label
1    2591
0    2591
Name: count, dtype: int64

## Naive Bayes

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score

#initial nb models
gnb = GaussianNB()
mnb = MultinomialNB()
bnb = BernoulliNB()

#split data into train and test
X_train, X_test, y_train, y_test = train_test_split(clean_news_data.drop('real_label', axis=1), clean_news_data['real_label'], test_size = 0.3, random_state = 101)

#fit models
gnb.fit(X_train, y_train)
mnb.fit(X_train, y_train)
bnb.fit(X_train, y_train)

#model predictions
y_pred_gnb = gnb.predict(X_test)
y_pred_mnb = mnb.predict(X_test)
y_pred_bnb = bnb.predict(X_test)

#show model results
print('GaussianNB Accuracy: ', accuracy_score(y_test, y_pred_gnb))
print('MultinomialNB Accuracy: ', accuracy_score(y_test, y_pred_mnb))
print('BernoulliNB Accuracy: ', accuracy_score(y_test, y_pred_bnb))

GaussianNB Accuracy:  0.8861736334405145
MultinomialNB Accuracy:  0.9434083601286174
BernoulliNB Accuracy:  0.9607717041800643


In [9]:
#gaussian nb grid search
from sklearn.model_selection import GridSearchCV

gnb_params = {'var_smoothing': [1, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9, 1e-10, 1e-11, 1e-12, 1e-13]}

gnb_grid = GridSearchCV(gnb, gnb_params, cv=5, n_jobs=-1, verbose=1)

gnb_grid.fit(X_train, y_train)

print('GaussianNB Best Score: ', gnb_grid.best_score_)
print('GaussianNB Best Params: ', gnb_grid.best_params_)

Fitting 5 folds for each of 14 candidates, totalling 70 fits
GaussianNB Best Score:  0.9101221620594661
GaussianNB Best Params:  {'var_smoothing': 1e-06}


In [10]:
#multinomial nb grid search
mnb_params = {'alpha': [1, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9, 1e-10, 1e-11, 1e-12, 1e-13]}

mnb_grid = GridSearchCV(mnb, mnb_params, cv=5, n_jobs=-1, verbose=1)

mnb_grid.fit(X_train, y_train)

print('MultinomialNB Best Score: ', mnb_grid.best_score_)
print('MultinomialNB Best Params: ', mnb_grid.best_params_)

Fitting 5 folds for each of 14 candidates, totalling 70 fits


MultinomialNB Best Score:  0.9434824736392133
MultinomialNB Best Params:  {'alpha': 1}


In [11]:
#bernoulli nb grid search
bnb_params = {'alpha': [1, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9, 1e-10, 1e-11, 1e-12, 1e-13]}

bnb_grid = GridSearchCV(bnb, bnb_params, cv=5, n_jobs=-1, verbose=1)

bnb_grid.fit(X_train, y_train)

print('BernoulliNB Best Score: ', bnb_grid.best_score_)
print('BernoulliNB Best Params: ', bnb_grid.best_params_)

Fitting 5 folds for each of 14 candidates, totalling 70 fits


c:\Users\voldy\Desktop\college stuff\CS350\venv\lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
2 fits failed out of a total of 70.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
2 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\voldy\Desktop\college stuff\CS350\venv\lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\voldy\Desktop\college stuff\CS350\venv\lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\voldy\Desktop\college stuff\CS350\venv\lib\site-packages\sklearn\naive_bayes.py", lin

BernoulliNB Best Score:  0.9608522846015009
BernoulliNB Best Params:  {'alpha': 1}


In [12]:
#split data into train and test
X_train, X_test, y_train, y_test = train_test_split(clean_news_data_ctx.drop('real_label', axis=1), clean_news_data['real_label'], test_size = 0.3, random_state = 101)

#fit models
gnb.fit(X_train, y_train)
mnb.fit(X_train, y_train)
bnb.fit(X_train, y_train)

#model predictions
y_pred_gnb = gnb.predict(X_test)
y_pred_mnb = mnb.predict(X_test)
y_pred_bnb = bnb.predict(X_test)

#show model results
print('GaussianNB Accuracy: ', accuracy_score(y_test, y_pred_gnb))
print('MultinomialNB Accuracy: ', accuracy_score(y_test, y_pred_mnb))
print('BernoulliNB Accuracy: ', accuracy_score(y_test, y_pred_bnb))

GaussianNB Accuracy:  0.8636655948553055
MultinomialNB Accuracy:  0.9337620578778135
BernoulliNB Accuracy:  0.9607717041800643


In [13]:
#gaussian nb grid search
from sklearn.model_selection import GridSearchCV

gnb_params = {'var_smoothing': [1, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9, 1e-10, 1e-11, 1e-12, 1e-13]}

gnb_grid = GridSearchCV(gnb, gnb_params, cv=5, n_jobs=-1, verbose=1)

gnb_grid.fit(X_train, y_train)

print('GaussianNB Best Score: ', gnb_grid.best_score_)
print('GaussianNB Best Params: ', gnb_grid.best_params_)

Fitting 5 folds for each of 14 candidates, totalling 70 fits
GaussianNB Best Score:  0.8814477058991166
GaussianNB Best Params:  {'var_smoothing': 0.0001}


In [14]:
#multinomial nb grid search
mnb_params = {'alpha': [1, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9, 1e-10, 1e-11, 1e-12, 1e-13]}

mnb_grid = GridSearchCV(mnb, mnb_params, cv=5, n_jobs=-1, verbose=1)

mnb_grid.fit(X_train, y_train)

print('MultinomialNB Best Score: ', mnb_grid.best_score_)
print('MultinomialNB Best Params: ', mnb_grid.best_params_)

Fitting 5 folds for each of 14 candidates, totalling 70 fits
MultinomialNB Best Score:  0.9360387574807637
MultinomialNB Best Params:  {'alpha': 1}


In [15]:
#bernoulli nb grid search
bnb_params = {'alpha': [1, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9, 1e-10, 1e-11, 1e-12, 1e-13]}

bnb_grid = GridSearchCV(bnb, bnb_params, cv=5, n_jobs=-1, verbose=1)

bnb_grid.fit(X_train, y_train)

print('BernoulliNB Best Score: ', bnb_grid.best_score_)
print('BernoulliNB Best Params: ', bnb_grid.best_params_)

Fitting 5 folds for each of 14 candidates, totalling 70 fits


c:\Users\voldy\Desktop\college stuff\CS350\venv\lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
5 fits failed out of a total of 70.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
3 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\voldy\Desktop\college stuff\CS350\venv\lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\voldy\Desktop\college stuff\CS350\venv\lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\voldy\Desktop\college stuff\CS350\venv\lib\site-packages\sklearn\naive_bayes.py", lin

BernoulliNB Best Score:  0.9608522846015009
BernoulliNB Best Params:  {'alpha': 1}
